In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
%pip install unsloth
try:
    import unsloth
    print(f"{[unsloth]} is installed and enabled.")
except ImportError:
    print(f"{[unsloth]} is not installed or enabled.")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # <--- HIDES THE SECOND GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import re
import time
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch
import gc

from datasets import concatenate_datasets, load_dataset

gc.collect()
torch.cuda.empty_cache()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen3.5-2B",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 512,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1, # Reduced to save memory
    "gradient_accumulation_steps": 4, # Increased to maintain effective batch size
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/Qwen3.5-2B",
}

CONFIG

In [ ]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "data_files": "SVGX_SFT_GEN_51k.json",  # <--- NEW: Target the specific file
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    "xingxm/SVGX-SFT-1M"  # <--- NEW: Add it to the active list
]

In [ ]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


from datasets import load_dataset, concatenate_datasets

def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")

    # --- MODIFIED: Check if 'data_files' exists in the config ---
    if "data_files" in cfg:
        ds = load_dataset(dataset_id, data_files=cfg["data_files"], split=cfg["split"])
    else:
        ds = load_dataset(dataset_id, split=cfg["split"])
    # ------------------------------------------------------------

    print(f"  Initial rows: {len(ds)}")
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
        print(f"  Rows after sampling: {len(ds)}")

    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    print(f"  Rows after to_prompt_svg mapping: {len(ds)}")

    # Filter out samples where prompt or svg is empty
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"  {dataset_id}: {len(ds)} usable rows after filtering")
    return ds

In [ ]:
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(train_ds[0])

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=torch.float16, # <--- BACK TO 16-BIT
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "v_proj"], # <--- STRICTLY REDUCED
    use_gradient_checkpointing=True,
    random_state=SEED,
)

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# 0. CLEAR DEAD MEMORY FROM PREVIOUS CRASHES
import gc
torch.cuda.empty_cache()
gc.collect()

# Calculate warmup steps
total_train_steps = CONFIG["num_train_epochs"] * (len(train_text) / 1 / (CONFIG["gradient_accumulation_steps"] * 2))
warmup_steps = round(CONFIG["warmup_ratio"] * total_train_steps)

# 1. APPLY THE STRICT FILTER
print("--- APPLYING STRICT LENGTH FILTER ---")
def is_safe_length(example):
    tokenized_len = len(tokenizer(text=example["text"])["input_ids"])
    return tokenized_len < (CONFIG["max_seq_length"] - 50)

original_len = len(train_text)
train_text = train_text.filter(is_safe_length, num_proc=1)
new_len = len(train_text)
print(f"Dropped {original_len - new_len} sequences that were too long.")

pure_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

# 3. SETUP TRAINING ARGUMENTS (MEMORY FIXES APPLIED)
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=1,                                         # <--- REDUCED TO 1 TO PREVENT OOM
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"] * 2, # <--- DOUBLED TO MAINTAIN BATCH SIZE
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, 
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=warmup_steps, 
    weight_decay=CONFIG["weight_decay"],
    fp16=True,                                                             # <--- FORCE FP16
    bf16=False,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_drop_last=True,
    average_tokens_across_devices=False, # <--- ADD THIS LINE
    optim="adamw_8bit", # Removed the "paged_" prefix
)

# 4. INITIALIZE TRAINER 
trainer = SFTTrainer(
    model=model,
    tokenizer=pure_tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,              
    dataset_num_proc=1,
    args=training_args,
)

# 5. TRAIN!
print("Starting training with Memory Fixes...")
train_result = trainer.train()
train_result

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
!zip -r model.zip /kaggle/working/Qwen3.5-2B